# 05 — Fine-tuning do assistente clínico

**Entrada:** `../data/{train,val,test}.jsonl` (saída do `04`)
**Saída:** adapter LoRA em `../modelos/lora_model`

Segue o fluxo da Aula 02: Unsloth + QLoRA 4-bit + `SFTTrainer`. Roda em Colab (T4) ou
em GPU local com pelo menos 8 GB.

Uma diferença em relação à aula: lá o dataset precisava ser convertido para o formato
alpaca, com um parser de tags `[|News|]`. Aqui o `04` já entrega no formato `messages`
(system/user/assistant), que é o padrão que o chat template do modelo consome direto.

## Instalação

No Colab, rodar uma vez por sessão.

In [ ]:
# !pip install -q unsloth
# !pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

# Config do treino
MODELO_BASE = "unsloth/Qwen3.5-4B"   # 9.3 GB de download; o 9B (19.3 GB) nao cabe em 8 GB
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
SEED = 3407

# Caminhos. No Colab, apontar para a pasta do Drive.
from pathlib import Path
DATA = Path("../data")
SAIDA = Path("../modelos")
SAIDA.mkdir(exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Dataset

O `04` gerou os três splits já separados por `cluster_id`, então perguntas parecidas
não aparecem em treino e teste ao mesmo tempo. Aqui é só carregar — não refazemos o
split, senão o cuidado do `04` se perde.

O campo `resposta_reescrita` marca as linhas cuja resposta foi reescrita por LLM.
Usamos ele na avaliação, para separar o que é texto de médico do que é texto gerado.

In [ ]:
ds = load_dataset("json", data_files={
    "train": str(DATA / "train.jsonl"),
    "val":   str(DATA / "val.jsonl"),
    "test":  str(DATA / "test.jsonl"),
})

for split in ds:
    d = ds[split]
    humano = 1 - sum(d["resposta_reescrita"]) / len(d)
    print(f"{split:<6} {len(d):>7,} linhas | resposta de médico: {humano:.0%}")

# Exemplo do formato que vamos treinar
print("\n--- exemplo ---")
for m in ds["train"][0]["messages"]:
    print(f"[{m['role']}] {m['content'][:150]}")

## Modelo base

QLoRA de 4 bits, como na aula: os pesos ficam quantizados e só o adapter LoRA treina em
precisão cheia. É isso que permite treinar numa GPU de 8 GB.

**Por que o 4B e não o 9B.** O 9B em 4 bits são ~5,5 GB só de pesos; somando ativações e
o que o sistema já usa da placa, não sobra espaço numa RTX 4060 de 8 GB. O 4B resolve com
folga. Download: 9,3 GB contra 19,3 GB.

### O `target_modules` merece atenção

A aula usava Llama-3, um transformer clássico onde toda camada tem `q_proj/k_proj/v_proj/
o_proj`. O Qwen3.5 é **híbrido**: das 33 camadas de texto, só 9 usam atenção clássica; as
outras 24 são *Gated DeltaNet*, com projeções de nomes diferentes.

Copiar a lista da aula não dá erro — o PEFT encontra os nomes e treina. Mas adaptaria a
mistura de tokens em apenas 9 de 33 camadas, deixando 24 intocadas. Por isso a lista
abaixo inclui as projeções do DeltaNet.

| bloco | camadas | módulos |
|---|---|---|
| `self_attn` | 9 | `q_proj` `k_proj` `v_proj` `o_proj` |
| `linear_attn` (DeltaNet) | 24 | `in_proj_qkv` `in_proj_a` `in_proj_b` `in_proj_z` `out_proj` |
| `mlp` | 33 | `gate_proj` `up_proj` `down_proj` |

O `conv1d` do DeltaNet fica de fora por não ser camada linear. A torre de visão do modelo
também não é tocada — treinamos só texto.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODELO_BASE,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",                          # atencao (9 camadas)
        "in_proj_qkv", "in_proj_a", "in_proj_b", "in_proj_z", "out_proj",  # DeltaNet (24)
        "gate_proj", "up_proj", "down_proj",                             # MLP (33)
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

In [ ]:
# Confere que a lista pegou os dois tipos de camada, e nao so a MLP.
from collections import Counter
alcancados = Counter()
for nome, _ in model.named_modules():
    if "lora_A" in nome:
        alcancados[nome.split(".lora_A")[0].split(".")[-1]] += 1
print("módulos com adapter LoRA:")
for n, c in alcancados.most_common():
    print(f"  {n:<16} {c:>4}")

## Formatação

O chat template do modelo transforma a lista `messages` no texto com os marcadores de
turno que ele espera. Na aula isso era um f-string montado à mão (o `alpaca_prompt`);
aqui usamos o template do próprio tokenizer, que é mais seguro — se o formato do modelo
mudar, o template acompanha.

Repare na saída abaixo: o Qwen3.5 insere um bloco `<think></think>` vazio antes da
resposta. É o formato dele para resposta direta, sem raciocínio passo a passo. O modelo
precisa aprender a emitir esse bloco, senão o formato quebra na hora de gerar — por isso
ele fica dentro da parte treinada, e vai aparecer nas respostas geradas mais adiante.

In [ ]:
def formata(exemplos):
    textos = [tokenizer.apply_chat_template(m, tokenize=False)
              for m in exemplos["messages"]]
    return {"text": textos}

ds = ds.map(formata, batched=True)
print(ds["train"][0]["text"][:600])

## Treino

Um ajuste importante em relação à aula: `train_on_responses_only`.

Sem ele, o modelo é treinado para gerar a conversa inteira — inclusive a pergunta do
médico e o system prompt. Como o system prompt é idêntico em todas as linhas, ele vira
o texto mais repetido do dataset e o modelo gasta capacidade decorando ele.

Com o ajuste, a loss só conta os tokens da resposta, que é o que queremos que ele
aprenda a produzir.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Nota: a Aula 02 passa dataset_text_field, max_seq_length e packing direto no
# SFTTrainer, e usa TrainingArguments. Isso era a API do trl<0.9; na versao atual
# esses parametros vivem dentro do SFTConfig, e o tokenizer virou processing_class.
config = SFTConfig(
    output_dir = str(SAIDA / "outputs"),
    seed = SEED,

    dataset_text_field = "text",
    max_length = MAX_SEQ_LENGTH,
    packing = False,
    dataset_num_proc = 2,

    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,

    # Para um teste rapido, descomente max_steps e comente num_train_epochs.
    # max_steps = 60,
    num_train_epochs = 2,

    learning_rate = 2e-4,
    bf16 = torch.cuda.is_bf16_supported(),
    fp16 = not torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    logging_steps = 25,
    eval_strategy = "epoch",
    report_to = [],
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["val"],
    args = config,
)

# Treina so nos tokens da resposta, ignorando system e pergunta.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# Confere que a mascara funcionou: o texto abaixo tem de ser so a resposta.
exemplo = trainer.train_dataset[0]
visiveis = [t for t in exemplo["labels"] if t != -100]
print(tokenizer.decode(visiveis))

In [ ]:
stats = trainer.train()
print(stats.metrics)

## Teste

Duas coisas para mostrar na apresentação:

1. A **loss separada** por tipo de resposta. Se o modelo for muito melhor nas respostas
   reescritas por LLM, ele aprendeu o estilo do reescritor em vez de conteúdo clínico.
2. **Respostas geradas** lado a lado com a referência, para leitura humana. Loss baixa
   não garante que o modelo respeita o limite de nunca prescrever sem validação.

In [ ]:
for rotulo, subset in [
    ("resposta de médico", ds["test"].filter(lambda e: not e["resposta_reescrita"])),
    ("resposta reescrita", ds["test"].filter(lambda e: e["resposta_reescrita"])),
]:
    m = trainer.evaluate(eval_dataset=subset)
    print(f"{rotulo:<20} {len(subset):>6,} linhas | loss {m['eval_loss']:.4f}")

In [ ]:
FastLanguageModel.for_inference(model)

for exemplo in ds["test"].select(range(3)):
    msgs = exemplo["messages"]
    prompt = tokenizer.apply_chat_template(
        msgs[:-1], tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(prompt, return_tensors="pt").to("cuda")

    saida = model.generate(**entrada, max_new_tokens=300, use_cache=True)
    gerado = tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:],
                              skip_special_tokens=True)

    print("=" * 90)
    print(f"CONDIÇÃO: {exemplo['condition']}")
    print(f"\nPERGUNTA: {msgs[1]['content']}")
    print(f"\nREFERÊNCIA: {msgs[2]['content'][:350]}")
    print(f"\nMODELO: {gerado[:350]}")

## Salvar

O adapter LoRA são poucos MB — é só a diferença em relação ao modelo base.

A última célula exporta em GGUF, que é o formato para rodar fora do Python. Vale saber
que hoje o Ollama não carrega GGUF de Qwen3.5; se der erro ali, o caminho é servir pelo
`llama-server` do llama.cpp.

In [ ]:
model.save_pretrained(str(SAIDA / "lora_model"))
tokenizer.save_pretrained(str(SAIDA / "lora_model"))
print(f"adapter salvo em {SAIDA / 'lora_model'}")

In [ ]:
# Merge do adapter com a base e conversao para GGUF.
model.save_pretrained_gguf(str(SAIDA / "gguf"), tokenizer, quantization_method="q4_k_m")

## Publicar no Hugging Face

Três formatos possíveis, e não precisa ser só um:

| formato | tamanho | para quê |
|---|---|---|
| adapter LoRA | ~100 MB | quem já tem o modelo base; é a entrega mais honesta do que treinamos |
| merged 16-bit | ~19 GB | quem quer carregar direto, sem juntar nada |
| GGUF q4_k_m | ~6 GB | rodar fora do Python (llama.cpp) |

**Sobre privacidade e licença.** O card do MedPT não declara licença, e ausência de
licença não é permissão. Como o modelo foi treinado nesses dados, ele é um derivado —
a mesma ressalva que travou a publicação do dataset no `04` vale aqui. Por isso
`PRIVADO = True`, e o card do modelo precisa citar o paper do MedPT e dizer que parte
das respostas de treino foi reescrita por LLM.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN ausente no .env")

REPO_MODELO = "emidiosouza/assistente-maternidade"   # ajuste
PRIVADO = True
PUBLICAR = False        # vire para True quando decidir publicar

print(f"destino: {REPO_MODELO} (privado={PRIVADO}) | PUBLICAR={PUBLICAR}")

In [ ]:
# Adapter LoRA: leve e rapido, e o que de fato foi treinado.
if PUBLICAR:
    model.push_to_hub(REPO_MODELO, token=HF_TOKEN, private=PRIVADO)
    tokenizer.push_to_hub(REPO_MODELO, token=HF_TOKEN, private=PRIVADO)
    print(f"adapter publicado em {REPO_MODELO}")
else:
    print("PUBLICAR=False — nada enviado.")

In [ ]:
# Modelo completo (base + adapter) e GGUF. Upload demorado: ~19 GB e ~6 GB.
if PUBLICAR:
    model.push_to_hub_merged(f"{REPO_MODELO}-merged", tokenizer,
                             save_method="merged_16bit",
                             token=HF_TOKEN, private=PRIVADO)
    model.push_to_hub_gguf(f"{REPO_MODELO}-gguf", tokenizer,
                           quantization_method="q4_k_m",
                           token=HF_TOKEN, private=PRIVADO)
    print("merged e GGUF publicados")
else:
    print("PUBLICAR=False — nada enviado.")